In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

In [2]:
from verimon import loaders
from random import randrange
from math import sqrt

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 4**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [3]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
show(mc_sv)

Output()

Output()

In [6]:
from verimon.MonitorLearning import learn_monitor

learned_monitor = learn_monitor(
    mc, 
    "init", 
    ["init", "normal", "snake", "ladder"], 
    'P>0.9 [ "good" ]', 
    0.7, 
    walks_per_state= 1000, 
    walk_len=100
)

Hypothesis 1: 1 states.
Hypothesis 2: 11 states.
Hypothesis 3: 12 states.
Hypothesis 4: 16 states.
Hypothesis 5: 17 states.
Hypothesis 6: 20 states.
Hypothesis 7: 26 states.
Hypothesis 8: 33 states.
Hypothesis 9: 34 states.
Hypothesis 10: 35 states.
-----------------------------------
Learning Finished.
Learning Rounds:  10
Number of states: 35
Time (in seconds)
  Total                : 9.25
  Learning algorithm   : 0.15
  Conformance checking : 9.1
Learning Algorithm
 # Membership Queries  : 1223
 # MQ Saved by Caching : 531
 # Steps               : 16302
Equivalence Query
 # Membership Queries  : 35000
 # Steps               : 2130912
-----------------------------------


In [57]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.unrolling import simulator_unroll, prune_monitor

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
mon = simulator_unroll(mon_cycl, 12)
prune_monitor(mon)
show(mon)

Output()

Output()

In [9]:
gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

In [63]:
from verimon.MDP_product import MC_MON_Product
from time import time

t = time()
pomdp = MC_MON_Product(mc_sv, mon, gb, "good")
# pomdp.apply_spec('P>=0.5 [ F<=3 "good"]')
pomdp.create_product(use_step_label=True)
pomdp.remove_unreachable_states()
print(f"Starting Paynt after {time() - t}s\n--------------------")
assignment = pomdp.check_paynt_prop('Pmax=? [ (F "goal") ]', 0.2)
# show(pomdp.pomdp)

Remove 2529 unreachable states
Starting Paynt after 26.01265811920166s
--------------------
2024-10-15 17:25:39,047 - pomdp.py - constructed POMDP having 19 observations.
2024-10-15 17:25:39,049 - pomdp.py - unfolding 1-FSC template into POMDP...
2024-10-15 17:25:39,051 - pomdp.py - constructed quotient MDP having 295 states and 1203 actions.
2024-10-15 17:25:39,052 - statistic.py - synthesis initiated, design space: 1e10
> progress 0.001%, elapsed 3 s, estimated 280295 s (3 days), iters = {MDP: 641}, opt = 0.9608
> progress 0.015%, elapsed 6 s, estimated 38834 s (10 hours), iters = {MDP: 1564}, opt = 0.9785
> progress 0.023%, elapsed 9 s, estimated 38159 s (10 hours), iters = {MDP: 2524}, opt = 0.9785
> progress 0.037%, elapsed 12 s, estimated 32163 s (8 hours), iters = {MDP: 3482}, opt = 0.9785
> progress 0.048%, elapsed 15 s, estimated 31117 s (8 hours), iters = {MDP: 4357}, opt = 0.9785
> progress 0.062%, elapsed 18 s, estimated 28919 s (8 hours), iters = {MDP: 5230}, opt = 0.9875


In [64]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

induced_mc, poss = pomdp.simulate_paynt_assignment(assignment, 1000)
player_path = poss
# player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in pomdp.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

2024-10-15 17:35:33,358 - pyplot.py - Loaded backend notebook version unknown.
s0, labels=!g0 init !s0 !l0 [pos=0]

	--[0, 1:normal]-->
s7, labels=!g0 !l1 !s6 [pos=6] normal
	--[1, 1:normal]-->
s14, labels=!g0 !l3 !s12 [pos=12] normal
	--[2, 1:normal]-->
s28, labels=!g1 !l7 !s16 [pos=16] good normal
	--[3, 1:normal]-->
s47, labels=!g1 !l14 !s16 [pos=16] good normal
	--[4, 1:normal]-->
s78, labels=!g1 !l25 !s16 [pos=16] good normal
	--[5, 1:normal]-->
s122, labels=!g1 !l39 !s16 [pos=16] accepting good normal
	--[6, 1:normal]-->
s178, labels=!g1 !l54 !s16 [pos=16] accepting good normal
	--[8, 1:normal]-->
s225, labels=!g1 !l66 !s16 [pos=16] accepting good normal
	--[10, 1:normal]-->
s258, labels=!g1 !l74 !s16 [pos=16] accepting good normal
	--[12, 1:normal]-->
s281, labels=!g1 !l79 !s16 [pos=16] accepting good normal
	--[14, 1:normal]-->
s294, labels=!g1 !l82 !s16 [pos=16] accepting good horizon normal
	--[16, 4:end]-->
s1, labels=goal
it took 3 tries until the goal was reached
Result of

<IPython.core.display.Javascript object>

2024-10-15 17:35:33,466 - animation.py - Animation.save using <class 'matplotlib.animation.HTMLWriter'>
2024-10-15 17:35:33,467 - animation.py - frame size in pixels is 600 x 600
2024-10-15 17:35:33,731 - animation.py - MovieWriter: clearing temporary path=<TemporaryDirectory '/tmp/tmpicqz47pg'>


In [52]:
from stormvogel.show import show
from stormvogel.mapping import stormpy_to_stormvogel
from stormpy import model_checking, parse_properties

imc =  stormpy_to_stormvogel(induced_mc)
result_goal = model_checking(induced_mc, parse_properties('Pmax=? [F "goal"]')[0])
result_stop = model_checking(induced_mc, parse_properties('Pmax=? [F "stop"]')[0])
prop_goal = result_goal.at(induced_mc.initial_states[0])
prop_stop = result_stop.at(induced_mc.initial_states[0])
print(f"probability to reach goal={prop_goal} and stop={prop_stop}. Total={prop_goal+prop_stop}")
show(imc)


probability to reach goal=0.7534882641611523 and stop=0.246511735838847. Total=0.9999999999999993


Output()

Output()